# RAG-SFAR — Exploration, Ablation & Démo sur LibreDataHub

**Datathon SFAR × InterHop 2026 — Projet `rag-sfar`**

Ce notebook permet de reproduire en 1 clic sur LibreDataHub :
1. L'audit forensique des **264 recommandations SFAR** (2000–2026) et du graphe de supersession temporelle.
2. L'inspection du parseur de **2 056 tableaux HTML** (`rowspan`/`colspan` + héritage des tableaux sans en-tête + Row-Group Cards + liaison des encadrés allergie).
3. La comparaison scientifique (**ablation C0 → C8 Frontier 2026**) sur les **171 questions Antibioprophylaxie** et les **30 questions Multi-RFE 2024–2026**.
4. L'interrogation en direct via le LLM local LibreDataHub (Ollama / vLLM) + vérification anti-hallucination.

In [ ]:
from pathlib import Path
import json
from rag_sfar.ingestion.loader import load_sfar_corpus
from rag_sfar.retrieval.vector_store import LocalVectorStore
from rag_sfar.retrieval.embedder import build_embedder
from rag_sfar.retrieval.reranker import build_reranker
from rag_sfar.retrieval.pipeline import SfarRagRetriever
from rag_sfar.config import RetrievalConfig, Settings

ROOT = Path('..').resolve()
docs = load_sfar_corpus(ROOT / 'data/sfar_recommendations/output')
print(f'Documents uniques enrichis : {len(docs)}')
print(f'Éditions historiques remplacées détectées : {sum(not d.metadata.is_latest_edition for d in docs)}')
print(f'Sous-domaines manquants imputés : {sum(d.metadata.subdomain_was_imputed for d in docs)}')

In [ ]:
# Charger l'index SOTA pré-construit
idx_dir = ROOT / 'index/table_aware_sota'
store = LocalVectorStore.load(idx_dir)
embedder = build_embedder(
    provider=store.manifest['embedding_provider'],
    model_name=store.manifest['embedding_model'],
    index_dir=idx_dir,
)
reranker = build_reranker('cross_encoder', 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
retriever = SfarRagRetriever(store=store, embedder=embedder, reranker=reranker)
print('Index prêt :', store.manifest)

In [ ]:
# Tester des requêtes cliniques pièges (allergie split-chunk, AOD 2026 vs urgence 2024, diff 2018 vs 2026)
queries = [
    'Quelle antibioprophylaxie pour une PTH par voie antérieure chez un patient allergique aux bêtalactamines ?',
    'Quel est le délai d\'arrêt des AOD avant une neurochirurgie intracrânienne programmée ?',
    'Patient sous anticoagulant avec hémorragie intracrânienne en urgence : faut-il attendre les tests biologiques ?',
    'Antibiothérapie probabiliste d\'une infection intra-abdominale communautaire sans sepsis ?',
    'Qu\'est-ce qui a changé dans l\'antibioprophylaxie SFAR entre 2018 et 2026 ?',
]
cfg = RetrievalConfig()
for q in queries:
    res, qa, timings = retriever.retrieve(q, config=cfg)
    print('=' * 80)
    print('Q:', q)
    print(f'Router: temporal={qa.temporal_mode} | setting={qa.setting} | modifiers={qa.modifiers} | {timings["total_retrieval_ms"]}ms')
    for i, r in enumerate(res[:4], 1):
        c = r.chunk
        tag = 'EN VIGUEUR' if c.is_latest_edition else f'ANTÉRIEURE ({c.effective_year})'
        exp = f' [+{r.expansion_reason}]' if r.is_expanded_context else ''
        print(f'  [{i}] ({c.effective_year} | {tag} | {c.chunk_type}{exp}) {c.doc_title[:55]} :: {c.text[:130]}...')

In [ ]:
# Visualiser le rapport d'ablation sauvegardé (C0 -> C8)
from IPython.display import Markdown, display
report_md = (ROOT / 'reports/ablation_report.md').read_text(encoding='utf-8')
display(Markdown(report_md))

In [ ]:
# Brancher le LLM LibreDataHub (Ollama / vLLM) et tester une génération clinique sourcée + vérifiée
from rag_sfar.generation.client import OpenAICompatibleLLM, SfarRagEngine

# Adapter l'URL et le modèle au serveur LibreDataHub demain !
LLM_BASE_URL = 'http://localhost:11434/v1'
LLM_MODEL = 'qwen2.5:14b-instruct'

llm = OpenAICompatibleLLM(base_url=LLM_BASE_URL, api_key='ollama', model=LLM_MODEL)
engine = SfarRagEngine(retriever=retriever, llm=llm)
# Décommenter une fois Ollama/vLLM joignable :
# resp = engine.answer('Quelle antibioprophylaxie pour une PTH voie antérieure chez un patient allergique aux bêtalactamines ?')
# print(resp.answer)
# print('Verification:', resp.verification)